# Piedmontese–Italian parallel corpus from Wikipedia

Mines Italian–Piedmontese sentence pairs from Wikipedia articles that exist in both languages. Articles are paired through Wikidata, and sentences within each pair of articles are aligned with LaBSE.

`pmswiki_filtered.jsonl` + Italian Wikipedia → `wiki_mined/wiki_mined.csv`

The input is the output of §1 in the Wiki corpus notebook (filtered dump, raw wikitext). That notebook in turn uses `wiki_mined.csv` to remove mined sentences from the monolingual corpus, so the order is: Wiki corpus §1 → this notebook → Wiki corpus §2 onwards.

`wiki_mined.csv` (5,304 pairs) is the corpus the models were trained on. The version published on Hugging Face, [`wiki_mined_clean.csv`](https://huggingface.co/datasets/offelia39/pms-wiki-mined-corpora) (4,714 pairs), was cleaned afterwards of identical pairs and near-copies, non-Italian sources and non-Piedmontese targets.

A GPU runtime is recommended for LaBSE.

## §0 Setup

Paths assume everything lives in one Drive folder; change `BASE` if yours is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q sentence-transformers datasets nltk mwparserfromhell

In [ ]:
import json
import re
import warnings
from pathlib import Path

import mwparserfromhell
import nltk
import numpy as np
import pandas as pd
import requests
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
warnings.filterwarnings('ignore')

In [ ]:
BASE = Path("/content/drive/MyDrive/Tesi_piedmontese/")

PMS_JSONL  = BASE / "pmswiki_filtered.jsonl"
OUTPUT_CSV = BASE / "wiki_mined" / "wiki_mined.csv"

SIM_THRESHOLD = 0.85   # minimum LaBSE cosine similarity
MIN_WORDS     = 5      # sentence length limits, in words
MAX_WORDS     = 100

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

## §1 Piedmontese articles
in: `pmswiki_filtered.jsonl`

Articles are loaded as raw wikitext and cleaned one at a time during alignment (§3).

In [ ]:
def load_pms_articles(path):
    """Return {title: raw_text}."""
    articles = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            title = obj.get("title", "").strip()
            text  = obj.get("raw_text", "").strip()
            if title and text:
                articles[title] = text
    return articles

pms_articles = load_pms_articles(PMS_JSONL)
print(f"{len(pms_articles):,} Piedmontese articles")

## §2 Italian counterparts

Wikidata links each Piedmontese article to its Italian equivalent. The Italian texts are streamed from the Hugging Face copy of Italian Wikipedia (`wikimedia/wikipedia`, snapshot `20231101.it`), so there's no need to download the full dump. Streaming stops once every title is found; in practice a few thousand are never found (redirects, renamed or deleted pages), so it goes through the whole dump, which takes a while.

The Wikidata query returns the links as they are today, so a rerun may give slightly different counts. For the corpus in the paper: 60,465 links, 57,218 Italian articles found.

In [ ]:
query = """
SELECT ?pmstitle ?ittitle WHERE {
  ?sitelink_pms schema:about ?item ;
                schema:isPartOf <https://pms.wikipedia.org/> ;
                schema:name ?pmstitle .
  ?sitelink_it  schema:about ?item ;
                schema:isPartOf <https://it.wikipedia.org/> ;
                schema:name ?ittitle .
}
"""
r = requests.get(
    "https://query.wikidata.org/sparql",
    params={"query": query, "format": "json"},
    headers={"User-Agent": "PiedmonteseWikiMiner/1.0"},
    timeout=120,
)
bindings = r.json()["results"]["bindings"]
pms_it_links = {b["pmstitle"]["value"]: b["ittitle"]["value"] for b in bindings}
print(f"{len(pms_it_links):,} Piedmontese–Italian article links")

In [ ]:
def fetch_italian_articles(it_titles):
    """Stream Italian Wikipedia and collect the given titles. Returns {it_title: text}."""
    title_set = set(it_titles)
    found = {}

    ds = load_dataset("wikimedia/wikipedia", "20231101.it", split="train", streaming=True)

    scanned = 0
    for ex in ds:
        scanned += 1
        if ex["title"] in title_set:
            found[ex["title"]] = ex["text"]
            if len(found) == len(title_set):
                break
        if scanned % 200_000 == 0:
            print(f"  {scanned:,} scanned, {len(found):,} found")

    print(f"Found {len(found):,} of {len(title_set):,} Italian articles "
          f"(scanned {scanned:,})")
    return found

it_articles = fetch_italian_articles(pms_it_links.values())

## §3 Sentence alignment

Both sides are split into sentences: NLTK's Italian tokenizer for Italian; for Piedmontese, which has no tokenizer, line breaks and then `.`, `!`, `?`. Sentences outside 5–100 words are discarded. Each Piedmontese sentence is paired with the most similar Italian sentence in the same article, and the pair is kept if their LaBSE cosine similarity is at least 0.85.

The Piedmontese wikitext is cleaned with roughly the same function as in the Wiki corpus notebook. This one also drops `[[Figura:…]]` links, but doesn't remove `[= …]` blocks.

In [ ]:
labse = SentenceTransformer("sentence-transformers/LaBSE")   # ~1.8 GB on first download

In [ ]:
_SENT_END_RE = re.compile(r'(?<=[.!?])\s+')

def split_italian(text):
    return nltk.sent_tokenize(text, language="italian")

def split_piedmontese(text):
    sents = []
    for para in text.split("\n"):
        para = para.strip()
        if not para:
            continue
        for s in _SENT_END_RE.split(para):
            s = s.strip()
            if s:
                sents.append(s)
    return sents

def is_valid(sent):
    return MIN_WORDS <= len(sent.split()) <= MAX_WORDS

In [ ]:
def clean_wikitext(text):
    """Remove wiki markup from Piedmontese raw_text before sentence alignment."""

    wikicode = mwparserfromhell.parse(text)

    # Remove categories
    for node in wikicode.filter_wikilinks():
        if str(node.title).startswith("Categorìa:"):
            try:
                wikicode.remove(node)
            except ValueError:
                pass

    # Remove file/image links — includes [[Figura:...]], the Piedmontese-specific prefix
    nodes_to_remove = []
    for node in wikicode.filter_wikilinks():
        title = str(node.title).lower()
        if any(title.startswith(p) for p in (
            "file:", "image:", "immagine:", "fitxer:", "figura:"
        )):
            nodes_to_remove.append(node)
    for node in nodes_to_remove:
        try:
            wikicode.remove(node)
        except ValueError:
            pass

    cleaned = wikicode.strip_code(normalize=True, collapse=True)

    # URLs first (contain pipes/brackets that confuse later patterns)
    cleaned = re.sub(r'https?://\S+', '', cleaned)
    cleaned = re.sub(r'www\.\S+', '', cleaned)

    # Section headers (Piedmontese + multilingual)
    cleaned = re.sub(
        r'\b(?:Anliure esterne|Sit istitussional|Colegament estern|Arferiment|Vod autis|'
        r'Referenze|Nòte|Bibliografìa|Collegamenti esterni|External links|Références|See also|'
        r'Ligação externa|Ligações externas)\b',
        '', cleaned, flags=re.IGNORECASE
    )

    # Image caption directives
    cleaned = re.sub(
        r'\|\s*(?:thumb|thumbnail|frame|frameless|left|right|sinistra|destra|center|centre|miniatura)\s*\|',
        ' ', cleaned, flags=re.IGNORECASE
    )
    cleaned = re.sub(
        r'\b(?:thumb|thumbnail|frame|frameless|left|right|sinistra|destra|center|centre|miniatura)\s*\|+',
        '', cleaned, flags=re.IGNORECASE
    )
    cleaned = re.sub(r'\bthumb\b', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\bthumbnail\b', '', cleaned, flags=re.IGNORECASE)

    # Pixel dimensions
    cleaned = re.sub(r'\b\d+\s*px\b', '', cleaned, flags=re.IGNORECASE)

    # Orphaned pipes
    cleaned = re.sub(r'^\s*\|\|?\s+', '', cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r'\|\|+', '', cleaned)
    cleaned = re.sub(r'\|\s*([A-Za-zàèéìòùäëïöüâêîôûãõ])', r'\1', cleaned)
    cleaned = re.sub(r'\s+\|\s+', ' ', cleaned)
    cleaned = re.sub(r'\|\s+', '', cleaned)
    cleaned = re.sub(r'\s+\|', '', cleaned)

    # ISO language codes and bibliography references
    cleaned = re.sub(
        r'(?:C\Sdes ISO|ISO 639|Arferiment bibliogràfich|https?://www\.sil\.org).*',
        '', cleaned, flags=re.IGNORECASE
    )

    cleaned = " ".join(cleaned.split())
    return cleaned

In [ ]:
def align_article_pair(pms_title, pms_text, it_text, threshold=SIM_THRESHOLD):
    """Return aligned pairs (source = Italian, target = Piedmontese) for one article."""
    pms_sents = [s for s in split_piedmontese(clean_wikitext(pms_text)) if is_valid(s)]
    it_sents  = [s for s in split_italian(it_text) if is_valid(s)]   # Italian text is already clean

    if not pms_sents or not it_sents:
        return []

    enc = dict(batch_size=64, show_progress_bar=False,
               convert_to_numpy=True, normalize_embeddings=True)
    pms_embs = labse.encode(pms_sents, **enc)
    it_embs  = labse.encode(it_sents, **enc)

    sim = pms_embs @ it_embs.T   # cosine, since embeddings are normalised

    pairs = []
    for i, pms_s in enumerate(pms_sents):
        j = int(np.argmax(sim[i]))
        score = float(sim[i, j])
        if score >= threshold:
            pairs.append({
                "source":        it_sents[j],
                "target":        pms_s,
                "similarity":    round(score, 4),
                "article_title": pms_title,
            })
    return pairs

## §4 Mining
out: `wiki_mined/wiki_mined.csv`

Results are saved to Drive every 25 articles. If the runtime disconnects, run §0–§3 again and then this cell: it continues from the saved file. Articles that gave no pairs aren't recorded in the file, so these are aligned again after a restart.

In [ ]:
CHECKPOINT_EVERY = 25

if OUTPUT_CSV.exists():
    all_pairs   = pd.read_csv(OUTPUT_CSV).to_dict("records")
    done_titles = {p["article_title"] for p in all_pairs}
    print(f"Resuming: {len(done_titles):,} articles, {len(all_pairs):,} pairs already saved")
else:
    all_pairs, done_titles = [], set()

article_pairs = [(p, i) for p, i in pms_it_links.items()
                 if p in pms_articles and i in it_articles]
remaining = [(p, i) for p, i in article_pairs if p not in done_titles]
print(f"Article pairs: {len(article_pairs):,}, remaining: {len(remaining):,}")

for idx, (pms_title, it_title) in enumerate(remaining, 1):
    all_pairs.extend(align_article_pair(pms_title, pms_articles[pms_title], it_articles[it_title]))

    if idx % CHECKPOINT_EVERY == 0 or idx == len(remaining):
        pd.DataFrame(all_pairs).drop_duplicates(subset=["source", "target"]) \
          .to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    if idx % 1000 == 0 or idx == len(remaining):
        print(f"[{idx:>6,}/{len(remaining):,}]  {len(all_pairs):,} pairs")

## §5 Final file
in/out: `wiki_mined/wiki_mined.csv`

Duplicate pairs are removed, the pairs are sorted by similarity, and the file is saved again. For the paper corpus there were no duplicates: 5,304 pairs from 4,336 articles, with a mean similarity of 0.88.

In [ ]:
df = (pd.DataFrame(all_pairs)
        .drop_duplicates(subset=["source", "target"])
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True))
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"{len(df):,} pairs from {df['article_title'].nunique():,} articles")
df["similarity"].describe().round(4)

In [ ]:
pd.set_option("display.max_colwidth", 120)
df.sample(min(10, len(df)), random_state=42).reset_index(drop=True)